# n001 — LLM Agent Behaviour Analyser

This notebook aggregates and visualises LLM-generated annotations of agent behaviour across gridworld experiments.

**REQUIRES:** `001_llm_agent_analyser.py` to be run first.

**Outputs** (saved to `<ROOT>/plots/`):
- `001_agent_behaviors.pdf` — 3-panel horizontal bar chart of event, behavior, and emergence rates per agent

Two analyses are included:
1. **Annotation rates** — counts of tagged events, behaviours, and emergent properties per agent, grouped by experiment.
2. **Anthropologist word frequency** — most/least common words from free-text anthropologist notes across experiments.

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
from core.utils.analysis_utils import get_exp_folders
from core.utils import ROOT
import pandas as pd
from matplotlib.ticker import MultipleLocator


# force reload these everytime
import importlib
importlib.reload(importlib.import_module('analysis_scripts.plot_utils'))
from analysis_scripts.plot_utils import color_map, plot_params

plt.rcParams.update(plot_params)

In [ ]:
plot_params['axes.spines.top'] = True
plot_params['axes.spines.right'] = True
plt.rcParams.update(plot_params)

## Configuration

Set `EXP_BASE_NAMES` to the list of experiment base names you want to compare. All matching completed runs under `ROOT/logs/` are collected automatically.

- `PLOT_NAMES` — display-friendly labels (defaults to title-cased experiment name).
- `SHOW_STD` — toggle standard-deviation bands on time-series plots.
- `MODEL` — the annotation model whose outputs are loaded from `logs/<exp>/annotations/<MODEL>/`.

In [ ]:
EXP_BASE_NAMES = [] # TO DEFINE

# Change this if you want other names in the plot
PLOT_NAMES = {exp_name: exp_name.replace("_", " ").title() for exp_name in EXP_BASE_NAMES}

EXP_PATH = ROOT / "logs"
SHOW_STD = False
MODEL = 'claude-sonnet-4-5-20250929'

experiments = {}
for base_name in EXP_BASE_NAMES:
    experiments[base_name] = get_exp_folders(log_path=EXP_PATH, 
                                             exp_name=base_name, 
                                             only_completed=True)

## Load Annotations

For each experiment run, per-agent annotation files (`<agent_id>.json`) are read from the model's annotation directory. Three tag categories are counted:

- **events** — discrete occurrences (e.g. a specific action taken).
- **behaviors** — persistent patterns (e.g. exploration style).
- **emergence** — open-ended emergent keywords noted by the model.

Counts are stored in `agent_annotations_counters[base_name]` and normalised by the number of annotated agents at plot time. Any runs with missing annotation files are reported under `missing`.

In [ ]:
with open(ROOT / 'analysis_scripts' / "tags.json", "r") as f:
    tags = json.load(f)

event_tags = list(tags['agent_events'].keys())
behavior_tags = list(tags['agent_behavior'].keys())
emergence_tags = list(tags['agent_emergence'].keys())

missing = {}

# load annotations
agent_annotations_counters = {}
for base_name in experiments:
    event_counters = {tag: np.zeros(len(experiments[base_name])) for tag in event_tags}
    behavior_counters = {tag: np.zeros(len(experiments[base_name])) for tag in behavior_tags}
    emergence_counters = {tag: np.zeros(len(experiments[base_name])) for tag in emergence_tags}
    agents_counters = np.zeros(len(experiments[base_name]))
    for exp_idx, exp in enumerate(experiments[base_name]):
        annotations_path = EXP_PATH / exp / "annotations" / MODEL

        if not annotations_path.exists():
            print("No annotations for experiment:", exp)
            continue

        annotations = {}

        # load agent_names file
        agent_names = json.load(open(EXP_PATH / exp / "agent_names.json"))

        for agent_id in agent_names.keys():
            if (EXP_PATH / exp / 'agent_logs' / f"{agent_id}.jsonl").exists():
                agent_file = annotations_path / f"{agent_id}.json"
                if not agent_file.exists():
                    missing.setdefault(exp, []).append(agent_id)
                    continue
                
                agents_counters[exp_idx] += 1
                ann = json.load(open(annotations_path / agent_file))
                annotations[agent_id] = ann
                for event in ann.get('events', []):
                    event_name = event['event']
                    if event_name and event_name in event_counters:
                        event_counters[event_name][exp_idx] += 1
                for behavior in ann.get('behaviors', []):
                    behavior_name = behavior['behavior']
                    if behavior_name and behavior_name in behavior_counters:
                        behavior_counters[behavior_name][exp_idx] += 1

                emergence = ann.get('emergence', {})
                for emergence_kw in emergence.get('keywords', []):
                    if emergence_kw in emergence_counters:
                        emergence_counters[emergence_kw][exp_idx] += 1
                    else:
                        emergence_counters[emergence_kw] = np.zeros(len(experiments[base_name]))
                        emergence_counters[emergence_kw][exp_idx] = 1
    agent_annotations_counters[base_name] = {
        'events': event_counters,
        'behaviors': behavior_counters,
        'emergence': emergence_counters,
        "agents": agents_counters
    }

## Annotation Bar Chart

`plot_annotations_horizontal_bar` renders a horizontal grouped bar chart for one tag category. Bars show the **mean count per agent** across runs; tags are sorted by overall mean (highest at top). The figure below shows all three categories side-by-side and is saved to `ROOT/plots/001_agent_behaviors.pdf`.

In [ ]:
def plot_annotations_horizontal_bar(
    data,
    ax,
    ann_type,
    ann_tags,
    experiments,
    color_map,
    title_suffix="",
):
    """
    Horizontal grouped barplot of annotation rates (value = count / agents).

    Bars show mean across runs; error bars show standard error (SE).
    Tags are ordered (descending) by overall mean across ALL experiments.
    """
    # build long-form dataframe (normalized per-count by agents)
    rows = []
    exp_order = list(experiments.keys() if isinstance(experiments, dict) else experiments)

    for base_name in exp_order:
        comms = np.asarray(data[base_name]["agents"])
        for tag in ann_tags:
            vals = np.asarray(data[base_name][ann_type].get(tag, np.zeros(len(comms))))
            norm = vals / comms
            for v in norm:
                rows.append({"tag": tag, "experiment": base_name, "value": float(v)})

    df = pd.DataFrame(rows)

    # ---- NEW: order tags by overall mean across all experiments ----
    # (mean of all values for that tag, pooled across experiments and runs)
    tag_means = (
        df.groupby("tag", as_index=False)["value"]
          .mean()
          .sort_values("value", ascending=False)
    )
    ordered_tags = tag_means["tag"].tolist()

    # keep only tags that were requested (and in the same universe)
    ordered_tags = [t for t in ordered_tags if t in set(ann_tags)][::-1]
    # ---------------------------------------------------------------

    # grouped bars (dodge along y)
    group_width = 0.7
    n_experiments = len(exp_order)
    offsets = np.linspace(-group_width / 2, group_width / 2, n_experiments)
    bar_h = group_width / max(n_experiments, 1) * 0.9

    tag_index = {tag: i for i, tag in enumerate(ordered_tags)}

    # plot
    for tag in ordered_tags:
        df_t = df[df["tag"] == tag]
        i = tag_index[tag]

        for j, exp in enumerate(exp_order[::-1]):
            vals = df_t[df_t["experiment"] == exp]["value"].dropna().values
            if vals.size == 0:
                continue

            mean = float(np.mean(vals))
            se = float(np.std(vals, ddof=1) / np.sqrt(vals.size)) if vals.size > 1 else 0.0
            y = i + offsets[j]

            ax.barh(
                y=y,
                width=mean,
                height=bar_h,
                color=color_map[exp],
                alpha=0.6,
                edgecolor="none",
                zorder=2,
            )

    # axes formatting
    ax.set_ylabel("")
    ax.set_yticks(range(len(ordered_tags)))
    ann_tags_fmt = ["\n".join(tag.split("_")).capitalize() for tag in ordered_tags]
    if "Recorder" in ann_tags_fmt:
        ann_tags_fmt[ann_tags_fmt.index("Recorder")] = "Record\nkeeping"
    ax.set_yticklabels(ann_tags_fmt)

    # put the highest-mean tag at the top (optional; remove if you prefer bottom)
    # ax.invert_yaxis()

    ax.set_xlabel("Count per Agents")
    if ann_type in ['events', 'behaviors']:
        ann_type = ann_type[:-1]  # singular
    ax.set_title(f"{ann_type.capitalize()} {title_suffix}".rstrip())

    ax.xaxis.grid(alpha=0.3)
    ax.yaxis.grid(False)

    # horizontal separators between tags
    for i in range(len(ordered_tags) - 1):
        ax.axhline(y=i + 0.5, color="gray", linewidth=1, alpha=0.5)

    ax.set_ylim(-0.5, len(ordered_tags) - 0.5)

    # ---- dynamic x-grid spacing ----
    ax.relim()
    ax.autoscale_view()

    x_max = ax.get_xlim()[1]
    step = 0.2 if x_max < 2 else 1.0

    ax.xaxis.set_major_locator(MultipleLocator(step))
    ax.xaxis.grid(True, which="major", linestyle=":")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 7))

plot_annotations_horizontal_bar(data=agent_annotations_counters, ax=axes[0], ann_type='events', ann_tags=event_tags, experiments=experiments, color_map=color_map)
plot_annotations_horizontal_bar(data=agent_annotations_counters, ax=axes[1], ann_type='behaviors', ann_tags=behavior_tags, experiments=experiments, color_map=color_map)
plot_annotations_horizontal_bar(data=agent_annotations_counters, ax=axes[2], ann_type='emergence', ann_tags=emergence_tags, experiments=experiments, color_map=color_map)

import matplotlib.patches as mpatches
handles = [
    mpatches.Patch(
        facecolor=color_map[bn], edgecolor='none', alpha=0.6,
        label=PLOT_NAMES.get(bn, bn)
    )
    for bn in experiments
]

# Place legend in the white space between axes[0] and axes[1]
# Adjust the y value (~0.62–0.68) to fine-tune vertical placement
legend = fig.legend(
    handles=handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -.08),  # (x, y) in figure coords
    ncol=min(4, len(handles)),
    frameon=False,
)

# fig.subplots_adjust(hspace=1.3)
# plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.tight_layout()
plt.savefig(ROOT / 'plots' / '001_agent_behaviors.pdf', dpi=300,     bbox_inches="tight"
)
plt.show()

## Anthropologist Notes — Word Frequency

Loads free-text `anthropologist_notes.json` files across all runs for each experiment and counts word frequencies after removing common stopwords. Produces a side-by-side bar chart of the **top** and **least common** words for each experiment, useful for identifying recurring themes or rare observations in the qualitative notes.

In [ ]:
import re
from collections import Counter

def extract_text_from_notes(notes_obj):
    """
    Accepts list/dict/str and returns a single string.
    """
    if isinstance(notes_obj, str):
        return notes_obj
    if isinstance(notes_obj, list):
        return " ".join([extract_text_from_notes(x) for x in notes_obj])
    if isinstance(notes_obj, dict):
        return " ".join([extract_text_from_notes(v) for v in notes_obj.values()])
    return ""

stopwords = {
    "the","and","to","of","a","in","for","is","on","that","with","as","at","by",
    "an","be","it","are","this","was","from","or","but","not","have","has","had",
    "their","they","them","we","you","our","its","these","those","also","can",
    "could","would","should","may","might","into","over","about","within","during", "agent",
    "artifact", "artifacts", "energy", "food"
}

anthro_text_per_basename = {}

for base_name in experiments:
    collected_texts = []
    for exp in experiments[base_name]:
        annotations_path = EXP_PATH / exp / "annotations" / MODEL
        notes_file = annotations_path / "anthropologist_notes.json"
        if not notes_file.exists():
            continue
        try:
            notes_obj = json.load(open(notes_file))
        except Exception:
            continue
        collected_texts.append(extract_text_from_notes(notes_obj))
    full_text = " ".join(collected_texts)
    anthro_text_per_basename[base_name] = full_text

def tokenize(text):
    # Lowercase, remove punctuation/numbers, split
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t and t not in stopwords and len(t) > 2]
    return tokens

TOP_N = 30

for base_name, text in anthro_text_per_basename.items():
    tokens = tokenize(text)
    if not tokens:
        print(f"[warn] No tokens for {base_name}")
        continue

    freq_all = Counter(tokens)
    top_freq = freq_all.most_common(TOP_N)
    least_freq = sorted(freq_all.items(), key=lambda x: x[1])[:TOP_N]

    top_words, top_counts = zip(*top_freq) if top_freq else ([], [])
    least_words, least_counts = zip(*least_freq) if least_freq else ([], [])

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))

    # Top words
    bars = ax[0].barh(range(len(top_words)), top_counts, color=color_map.get(base_name, "gray"))
    ax[0].set_yticks(range(len(top_words)))
    ax[0].set_yticklabels(top_words)
    ax[0].invert_yaxis()
    ax[0].set_xlabel("Frequency")
    ax[0].set_title(f"Top {TOP_N} words: {base_name}")
    for i, b in enumerate(bars):
        ax[0].text(b.get_width() + max(top_counts)*0.01, b.get_y() + b.get_height()/2,
                   str(top_counts[i]), va="center", fontsize=8)

    # Least common words
    bars2 = ax[1].barh(range(len(least_words)), least_counts, color=color_map.get(base_name, "lightgray"))
    ax[1].set_yticks(range(len(least_words)))
    ax[1].set_yticklabels(least_words)
    ax[1].invert_yaxis()
    ax[1].set_xlabel("Frequency")
    ax[1].set_title(f"Least {TOP_N} words: {base_name}")
    for i, b in enumerate(bars2):
        ax[1].text(b.get_width() + (max(least_counts) if least_counts else 0)*0.05,
                   b.get_y() + b.get_height()/2, str(least_counts[i]), va="center", fontsize=8)

    plt.tight_layout()
    # out_path = ROOT / "plots" / f"001_anthropologist_wordfreq_{base_name}.pdf"
    # plt.savefig(out_path, dpi=150)
    # print(f"[saved] {out_path}")